# Anti-Money Laundering Detection on IBM AMLworld
## Gradient-Boosted Baseline vs. Multi-GNN (Multi-PNA+EU) Replication

**Goal.** Train an honest tabular baseline (XGBoost) for laundering detection on the
IBM AMLworld **HI-Small** dataset, then replicate the best published GNN configuration,
**Multi-PNA+EU** (PNA backbone + edge updates + reverse message passing + port numbering
+ ego IDs), from:

> Egressy et al., *Provably Powerful Graph Neural Networks for Directed Multigraphs* (AAAI 2024)
> [arXiv:2306.11586](https://arxiv.org/abs/2306.11586), code: [IBM/Multi-GNN](https://github.com/IBM/Multi-GNN)
> Dataset: Altman et al., *Realistic Synthetic Financial Transactions for AML Models* (NeurIPS 2023)
> [arXiv:2306.16424](https://arxiv.org/abs/2306.16424)

**Published targets we are trying to land near (minority-class F1, %):**

| Model | HI-Small | LI-Small |
|---|---:|---:|
| GIN | 28.70 +- 1.13 | 7.90 +- 2.78 |
| GIN+EU | 47.73 +- 7.86 | 20.62 +- 2.41 |
| PNA | 56.77 +- 2.41 | ~15-16 |
| GFP + XGBoost | 63.23 +- 0.17 | 27.30 +- 0.33 |
| **Multi-PNA+EU** | **68.16 +- 2.65** | **33.07 +- 2.63** |

**Runtime expectations (be realistic before you start):**
- Sections 1-4 (EDA + XGBoost baseline): ~20-30 min on a free Colab CPU/GPU runtime.
- Section 5 (GNN training): the paper reports ~7.7 h total training for plain PNA on a
  V100. Multi-PNA+EU roughly doubles per-epoch cost (two message-passing directions).
  Treat the full run as an **overnight job** on a Colab Pro GPU (L4/A100).
- Use `Runtime > Change runtime type > GPU` before Section 5.

**Environment**: built for Google Colab (Pro recommended for Section 5).

## 0. Setup

The IBM Multi-GNN repo pins `torch 2.0.1` + `torch-geometric 2.3.1`. PyG's companion
packages (`torch-scatter`, `torch-sparse`) are compiled C++/CUDA extensions, so their
wheels must match the torch version *exactly*. This is the number one source of
failures when reproducing the repo, which is why we pin everything here.

**Colab note:** Colab ships a newer torch. Installing the pinned version requires a
**runtime restart** after this cell finishes (`Runtime > Restart runtime`), then re-run
from the imports cell onward. You only pay this once per session.

In [ ]:
# --- Install pinned dependencies -------------------------------------------
# torch/PyG pins match IBM/Multi-GNN's env.yml. Do not "upgrade" these:
# mismatched torch-scatter wheels fail at import time with cryptic symbol errors.
!pip -q install torch==2.0.1 --index-url https://download.pytorch.org/whl/cu118
!pip -q install torch-geometric==2.3.1
!pip -q install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.1+cu118.html

# Baseline + utility stack (version-flexible, nothing exotic).
!pip -q install xgboost scikit-learn pandas matplotlib kagglehub

print("Install done. If torch was downgraded, RESTART THE RUNTIME now, then continue below.")

In [ ]:
# --- Imports and versions ---------------------------------------------------
import os, json, glob, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import xgboost as xgb
import sklearn

print("torch      :", torch.__version__, "| cuda:", torch.cuda.is_available())
print("xgboost    :", xgb.__version__)
print("sklearn    :", sklearn.__version__)
print("pandas     :", pd.__version__)

# One place to flip between dataset variants later (HI-Small first, always).
VARIANT = "HI-Small"          # develop here; treat LI-Small as the hard test later
SEED = 0
np.random.seed(SEED)

### 0.1 Download the dataset

We use `kagglehub`, which handles Kaggle auth via your API token
(`kaggle.json`). If you have not set that up: Kaggle > Account > Create New API Token,
then upload the file when prompted, or place it at `~/.kaggle/kaggle.json`.

Dataset page: [IBM Transactions for Anti Money Laundering](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml)

In [ ]:
# --- Download AMLworld from Kaggle (~1-2 GB for the Small variants) ---------
import kagglehub

DATA_DIR = kagglehub.dataset_download(
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
)
print("Dataset directory:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

CSV_PATH = os.path.join(DATA_DIR, f"{VARIANT}_Trans.csv")
assert os.path.exists(CSV_PATH), f"Expected {CSV_PATH}. Check the listing above."
print("Using:", CSV_PATH)

## 1. Load the data and look at it

HI-Small is roughly **5M transactions across ~515K accounts over 10 simulated days**,
with a laundering rate of about **1 in 981 transactions (~0.1%)**. Every row is a
payment between two accounts; `Is Laundering` is the ground-truth label, which the
simulator can provide *completely* (real banks never have complete labels, which is
exactly why this synthetic benchmark exists).

In [ ]:
# --- Load and normalize column names ----------------------------------------
# Raw columns: Timestamp, From Bank, Account, To Bank, Account.1, Amount Received,
# Receiving Currency, Amount Paid, Payment Currency, Payment Format, Is Laundering
df = pd.read_csv(CSV_PATH)
df.columns = ["ts", "from_bank", "from_acct", "to_bank", "to_acct",
              "amt_received", "cur_received", "amt_paid", "cur_paid",
              "fmt", "is_laundering"]

# Timestamps arrive as strings like "2022/09/01 00:20".
df["ts"] = pd.to_datetime(df["ts"], format="%Y/%m/%d %H:%M")

# Account IDs are only unique within a bank, so build globally unique node ids.
# (This matters later for graph features and for the GNN formatting step.)
df["from_id"] = df["from_bank"].astype(str) + "_" + df["from_acct"].astype(str)
df["to_id"]   = df["to_bank"].astype(str)   + "_" + df["to_acct"].astype(str)

print(df.shape)
df.head()

In [ ]:
# --- Headline statistics ------------------------------------------------------
n = len(df)
pos = int(df["is_laundering"].sum())
print(f"transactions      : {n:,}")
print(f"laundering        : {pos:,}  ({pos/n:.4%},  1 per {n//max(pos,1):,})")
print(f"unique accounts   : {pd.concat([df['from_id'], df['to_id']]).nunique():,}")
print(f"unique banks      : {pd.concat([df['from_bank'], df['to_bank']]).nunique():,}")
print(f"time span         : {df['ts'].min()}  ->  {df['ts'].max()}")
print(f"currencies        : {df['cur_paid'].nunique()}")
print(f"payment formats   : {list(df['fmt'].unique())}")

In [ ]:
# --- EDA: the four plots that define this problem ----------------------------
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# (a) Class imbalance. Log scale, otherwise the positive bar is invisible.
counts = df["is_laundering"].value_counts()
axes[0, 0].bar(["clean", "laundering"], counts.values, color=["#4878CF", "#D65F5F"])
axes[0, 0].set_yscale("log")
axes[0, 0].set_title(f"Class balance (log scale): 1 laundering per {n//max(pos,1):,} tx")
for i, v in enumerate(counts.values):
    axes[0, 0].text(i, v, f"{v:,}", ha="center", va="bottom")

# (b) Amount distribution, clean vs laundering. Laundering skews larger but overlaps
#     heavily with clean traffic: amount alone will never separate the classes.
bins = np.logspace(0, 7, 60)
axes[0, 1].hist(df.loc[df.is_laundering == 0, "amt_paid"].clip(1, 1e7), bins=bins,
                alpha=0.6, label="clean", density=True)
axes[0, 1].hist(df.loc[df.is_laundering == 1, "amt_paid"].clip(1, 1e7), bins=bins,
                alpha=0.6, label="laundering", density=True)
axes[0, 1].set_xscale("log"); axes[0, 1].legend()
axes[0, 1].set_title("Amount distribution (log x, density)")

# (c) Volume over time. The temporal axis is what our train/val/test split follows.
hourly = df.set_index("ts").resample("6H")["is_laundering"].agg(["size", "sum"])
axes[1, 0].plot(hourly.index, hourly["size"], label="all tx")
axes[1, 0].set_title("Transactions per 6h window"); axes[1, 0].legend()

# (d) Payment formats. Format is a strong contextual feature (cheque vs wire vs ACH).
df["fmt"].value_counts().plot(kind="barh", ax=axes[1, 1], color="#6ACC65")
axes[1, 1].set_title("Payment format counts")

plt.tight_layout(); plt.show()

**What the plots tell us before any modeling:**

1. **Accuracy is a useless metric here.** Predicting "clean" for every row scores 99.9%.
   Everything below is evaluated with minority-class F1, PR-AUC, and recall at a fixed
   alert budget.
2. **Single-transaction features overlap heavily** between classes. Laundering is a
   *behavioral and structural* pattern (velocity, counterparty structure), which is why
   feature engineering (Section 3) and graph structure (Section 5) carry the signal.
3. **The data is a time series.** Random splits would leak future behavior into
   training features and inflate every number. We split by time, next.

## 2. Temporal train / validation / test split (60 / 20 / 20)

We follow the paper's protocol exactly: order transactions by timestamp and cut at the
60th and 80th percentiles. Train on the earliest 60%, validate on the next 20%, test on
the final 20%. This mirrors reality: a bank trains on the past and scores the future.

**Never random-split this dataset.** Velocity and aggregate features computed on a
random split leak future information and produce dishonestly high scores.

In [ ]:
# --- Temporal split -----------------------------------------------------------
df = df.sort_values("ts").reset_index(drop=True)

t1 = df["ts"].quantile(0.60)   # end of train
t2 = df["ts"].quantile(0.80)   # end of validation

split = np.where(df["ts"] <= t1, "train", np.where(df["ts"] <= t2, "val", "test"))
df["split"] = split

summary = df.groupby("split").agg(
    rows=("is_laundering", "size"),
    laundering=("is_laundering", "sum"),
).reindex(["train", "val", "test"])
summary["rate"] = (summary["laundering"] / summary["rows"]).map("{:.4%}".format)
print(f"t1 = {t1}   t2 = {t2}")
summary

## 3. Feature engineering for the tabular baseline

We build ~15 features in three groups. The design constraint is **leakage safety**:
every feature for row *i* may only use information from rows strictly before *i* in
time. We get this by sorting once and using cumulative (expanding) statistics.

| Group | Features | Intuition |
|---|---|---|
| Transaction | log amount, round-amount flag, cross-bank flag, currency mismatch, payment format, hour, weekday | What does this payment look like on its own? |
| Sender history | prior tx count, prior mean amount, ratio to prior mean, seconds since last tx | Is this account behaving unusually vs. its own past? |
| Counterparty | receiver prior tx count, sender's distinct receivers so far | Early proxy for fan-out / mule structure |

This deliberately stops short of the paper's Graph Feature Preprocessor (cycle counts,
scatter-gather detection). That gap between our baseline and GFP+XGBoost (63.23) is
itself informative: it estimates how much of the signal is *structural*.

In [ ]:
# --- Leakage-safe feature construction (vectorized, ~5M rows in well under a minute per group)
X = df.copy()

# (1) Transaction-level features
X["log_amt"]        = np.log1p(X["amt_paid"])
X["round_amt"]      = (X["amt_paid"] % 100 == 0).astype(np.int8)
X["cross_bank"]     = (X["from_bank"] != X["to_bank"]).astype(np.int8)
X["cur_mismatch"]   = (X["cur_received"] != X["cur_paid"]).astype(np.int8)
X["fmt_code"]       = X["fmt"].astype("category").cat.codes
X["hour"]           = X["ts"].dt.hour.astype(np.int8)
X["dow"]            = X["ts"].dt.dayofweek.astype(np.int8)

# (2) Sender-history features. cumcount() counts STRICTLY EARLIER rows for the same
#     sender, so it is leakage-safe by construction on time-sorted data.
X["snd_prior_n"] = X.groupby("from_id").cumcount()
# Expanding mean of the sender's PAST amounts, excluding the current row:
#   (cumulative sum - current amount) / number of prior transactions
cum = X.groupby("from_id")["amt_paid"].cumsum()
X["snd_prior_mean"] = (cum - X["amt_paid"]) / X["snd_prior_n"].replace(0, np.nan)
X["amt_vs_prior"]   = (X["amt_paid"] / X["snd_prior_mean"]).fillna(1.0).clip(0, 1e4)
X["snd_gap_s"] = (X.groupby("from_id")["ts"].diff().dt.total_seconds()
                  .fillna(86400 * 30))          # "never seen before" -> large gap

# (3) Counterparty features
X["rcv_prior_n"] = X.groupby("to_id").cumcount()
# Distinct receivers this sender has paid so far (expanding nunique via first-seen flags):
first_pair = ~X.duplicated(subset=["from_id", "to_id"])
X["snd_distinct_rcv"] = first_pair.groupby(X["from_id"]).cumsum()

FEATURES = ["log_amt", "round_amt", "cross_bank", "cur_mismatch", "fmt_code",
            "hour", "dow", "snd_prior_n", "snd_prior_mean", "amt_vs_prior",
            "snd_gap_s", "rcv_prior_n", "snd_distinct_rcv"]

X[FEATURES].describe().T.round(2)

## 4. Baseline: XGBoost

Class imbalance is handled with `scale_pos_weight` (the negative:positive ratio in the
training window), the standard first move before anything fancier (undersampling,
focal loss). We early-stop on validation PR-AUC (`aucpr`), the right objective for a
0.1% positive rate.

The classification threshold is then **tuned on the validation set** to maximize
minority F1, and only that final threshold touches the test set, once.

In [ ]:
# --- Train XGBoost ------------------------------------------------------------
tr, va, te = (X[X.split == s] for s in ["train", "val", "test"])
Xtr, ytr = tr[FEATURES], tr["is_laundering"]
Xva, yva = va[FEATURES], va["is_laundering"]
Xte, yte = te[FEATURES], te["is_laundering"]

spw = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
print(f"scale_pos_weight = {spw:.0f}")

clf = xgb.XGBClassifier(
    n_estimators=600, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric="aucpr", early_stopping_rounds=30,
    tree_method="hist", n_jobs=-1, random_state=SEED,
)
clf.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=100)

In [ ]:
# --- Evaluate: threshold tuned on validation, reported on test -----------------
from sklearn.metrics import (precision_recall_curve, average_precision_score,
                             f1_score, classification_report)

p_va = clf.predict_proba(Xva)[:, 1]
p_te = clf.predict_proba(Xte)[:, 1]

# Sweep thresholds on VALIDATION only.
prec, rec, thr = precision_recall_curve(yva, p_va)
f1s = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
best = np.nanargmax(f1s[:-1])
t_star = thr[best]
print(f"chosen threshold (val): {t_star:.4f}  (val F1 = {f1s[best]:.4f})")

# Apply once to TEST.
yhat = (p_te >= t_star).astype(int)
test_f1 = f1_score(yte, yhat)
test_ap = average_precision_score(yte, p_te)
print(f"TEST minority-class F1 : {test_f1:.4f}")
print(f"TEST PR-AUC (avg prec) : {test_ap:.4f}")
print()
print(classification_report(yte, yhat, target_names=["clean", "laundering"], digits=4))

# Keep for the final comparison section.
RESULTS = {"XGBoost (ours, no graph features)": test_f1 * 100}

In [ ]:
# --- Visual diagnostics ---------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# (a) Precision-recall curve on test.
prec_t, rec_t, _ = precision_recall_curve(yte, p_te)
axes[0].plot(rec_t, prec_t)
axes[0].set_xlabel("recall"); axes[0].set_ylabel("precision")
axes[0].set_title(f"Test PR curve (AP = {test_ap:.3f})")

# (b) Recall at a fixed alert budget: the operational metric. If a risk team can only
#     investigate K alerts from this test window, what fraction of laundering is caught?
order = np.argsort(-p_te)
sorted_y = yte.values[order]
cum_recall = np.cumsum(sorted_y) / max(sorted_y.sum(), 1)
budgets = np.arange(1, len(sorted_y) + 1)
axes[1].plot(budgets, cum_recall)
axes[1].set_xscale("log")
for k in [100, 1000, 10000]:
    if k <= len(cum_recall):
        axes[1].scatter([k], [cum_recall[k - 1]], zorder=3)
        axes[1].annotate(f"{cum_recall[k-1]:.0%} @ {k:,}", (k, cum_recall[k - 1]),
                         textcoords="offset points", xytext=(6, -12))
axes[1].set_xlabel("alert budget (log)"); axes[1].set_ylabel("recall")
axes[1].set_title("Recall at fixed review capacity")

# (c) Feature importance: which signals carry the model?
imp = pd.Series(clf.feature_importances_, index=FEATURES).sort_values()
imp.plot(kind="barh", ax=axes[2], color="#4878CF")
axes[2].set_title("XGBoost feature importance (gain)")

plt.tight_layout(); plt.show()

**Reading the baseline honestly:**

- Do not expect to match GFP+XGBoost's published 63.23 F1. That configuration feeds the
  trees *graph* features (cycles, scatter-gather membership, vertex statistics) from
  IBM's Graph Feature Preprocessor. Our features see each transaction plus its sender's
  history, but they cannot see *structure*: a mule account receiving from 12 strangers
  looks locally normal.
- The gap between this number and the GNN result in Section 5 is the measured value of
  topology, which is the entire thesis of the papers we are replicating.
- The recall-at-budget curve is the number a risk team would actually act on. Quote it
  as "at K alerts per window, we catch X% of laundering", not as an abstract F1.

## 5. Replicating Multi-PNA+EU (IBM Multi-GNN)

Now the main event. `models.py` in the IBM repo defines four scaffolds (GINe, GATe,
PNA, RGCN); the paper's best AML configuration stacks four adaptations on PNA:

| Piece | Repo flag | What it adds |
|---|---|---|
| PNA backbone | `--model pna` | Multi-aggregator neighborhoods: mean/min/max/std x identity/amplification/attenuation scalers, 5 towers |
| Edge updates | `--emlps` | Each transaction edge re-computes its own embedding every layer (labels live on edges) |
| Reverse message passing | `--reverse_mp` | Information flows against edge direction too; the single biggest win in the ablations |
| Port numbering | `--ports` | Edges get per-node local indices so the model can count distinct counterparties |
| Ego IDs | `--ego` | Marks the center node of each sampled neighborhood (small effect on AML) |

Shared training protocol (from the papers): 2 GNN layers, hidden 20 for PNA
(lr 0.000612, dropout 0.083, minority-class weight 7.08, from `model_settings.json`),
neighbor sampling 100 x 100, the same 60/20/20 temporal split as Section 2, and
minority-class F1 as the metric.

**Published target: 68.16 +- 2.65 on HI-Small.** A single seed landing anywhere in
roughly 65.5 to 70.8 is a successful replication.

In [ ]:
# --- Clone the official repo ----------------------------------------------------
!git clone -q https://github.com/IBM/Multi-GNN.git
!ls Multi-GNN

In [ ]:
# --- (Optional but recommended) persist checkpoints to Google Drive --------------
# Colab WILL disconnect during a multi-hour run. Save outputs somewhere durable.
PERSIST = False   # flip to True on Colab
if PERSIST:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/aml_multignn_checkpoints"
    os.makedirs(CKPT_DIR, exist_ok=True)
    print("Checkpoints will be copied to:", CKPT_DIR)

In [ ]:
# --- Format the Kaggle CSV into the repo's expected layout ------------------------
# The repo ships format_kaggle_files.py exactly for this dataset.
!cd Multi-GNN && python format_kaggle_files.py "{CSV_PATH}"

# Locate the produced file (the script writes formatted_transactions.csv).
produced = (glob.glob("Multi-GNN/**/formatted_transactions.csv", recursive=True)
            + glob.glob(os.path.join(DATA_DIR, "**/formatted_transactions.csv"), recursive=True)
            + glob.glob("formatted_transactions.csv"))
assert produced, "formatted_transactions.csv not found; check the cell output above."
FORMATTED = produced[0]
print("Formatted file:", FORMATTED)

# Arrange the directory layout main.py expects: <data_root>/Small_HI/formatted_transactions.csv
os.makedirs("Multi-GNN/data/Small_HI", exist_ok=True)
target = "Multi-GNN/data/Small_HI/formatted_transactions.csv"
if os.path.abspath(FORMATTED) != os.path.abspath(target):
    os.replace(FORMATTED, target)
print("Data in place:", target, f"({os.path.getsize(target)/1e9:.2f} GB)")

In [ ]:
# --- Point data_config.json at our data root --------------------------------------
# Inspect the config first (structures occasionally change between repo commits),
# then set every path-like entry to our data directory.
cfg_path = "Multi-GNN/data_config.json"
cfg = json.load(open(cfg_path))
print("Before:", json.dumps(cfg, indent=2))

def set_paths(node, new_root):
    # Defensive: rewrite any string value that looks like a data path.
    if isinstance(node, dict):
        for k, v in node.items():
            if isinstance(v, str) and ("path" in k.lower() or "dir" in k.lower() or "data" in k.lower()):
                node[k] = new_root
            else:
                set_paths(v, new_root)

set_paths(cfg, os.path.abspath("Multi-GNN/data/"))
json.dump(cfg, open(cfg_path, "w"), indent=2)
print("After:", json.dumps(cfg, indent=2))

### 5.1 Sanity run first: plain GIN

Before spending a night of GPU on Multi-PNA+EU, prove the pipeline end to end with the
cheapest model. Two things to check in the output:

- Training proceeds and validation F1 is **in the neighborhood of the published 28.7**.
- If you see F1 above ~90, something leaked (almost always a broken split); if ~0,
  the class weight is not being applied. Fix before proceeding.

In [ ]:
# --- Sanity: plain GIN (published HI-Small F1: 28.70 +- 1.13) ----------------------
# Full run is ~6h on a V100-class GPU. You do NOT need to finish it: watch the first
# validation scores trend into a sane range, then interrupt (Runtime > Interrupt).
!cd Multi-GNN && python main.py --data Small_HI --model gin --tqdm

### 5.2 The replication run: Multi-PNA+EU

All four adaptation flags on. Expect an **overnight run**. If Colab disconnects, the
`--save_model` checkpoint plus `--finetune` lets you resume rather than restart.

If you hit CUDA OOM: PNA is the memory-heavy backbone (aggregators x scalers x towers,
doubled again by reverse message passing). First lever is the batch size in the repo
config; do not raise the hidden size, it is 20 for a reason.

In [ ]:
# --- Multi-PNA+EU (published HI-Small F1: 68.16 +- 2.65) --------------------------
!cd Multi-GNN && python main.py --data Small_HI --model pna \
    --emlps --reverse_mp --ports --ego \
    --save_model --unique_name multi_pna_eu_seed0 --tqdm

# Copy checkpoints to Drive if persistence is on.
if PERSIST:
    ckpts = glob.glob("Multi-GNN/**/*multi_pna_eu_seed0*", recursive=True)
    for f in ckpts:
        !cp -r "{f}" "{CKPT_DIR}/"
    print("Persisted:", ckpts)

In [ ]:
# --- Record YOUR test F1 from the run above ---------------------------------------
# Read the final test minority-class F1 from the training log printed by main.py
# and enter it here (as a percentage, e.g. 66.8). Left as None until you run it:
# no fabricated numbers in this notebook.
MY_MULTI_PNA_EU_F1 = None   # <-- fill in after the run completes

if MY_MULTI_PNA_EU_F1 is not None:
    RESULTS["Multi-PNA+EU (ours, 1 seed)"] = MY_MULTI_PNA_EU_F1

### 5.3 (Optional) The ablation ladder

The single most interview-worthy artifact from this project is not the final number,
it is the ladder showing what each adaptation buys. Each row is one training run;
spread them across sessions as budget allows.

| Run | Command suffix | Published F1 |
|---|---|---:|
| PNA | *(no flags)* | 56.77 |
| PNA + EU | `--emlps` | intermediate |
| + reverse MP | `--emlps --reverse_mp` | intermediate |
| + ports + ego | `--emlps --reverse_mp --ports --ego` | 68.16 |

The story the ladder tells: architecture matters less than *giving message passing the
information it provably cannot compute on its own* (direction-reversed context, the
ability to count distinct counterparties).

## 6. Results: published vs. ours

In [ ]:
# --- Comparison chart: published numbers vs our runs --------------------------------
published = {
    "GIN": 28.70,
    "RGCN": 41.78,
    "GIN+EU": 47.73,
    "PNA": 56.77,
    "GFP + LightGBM": 62.86,
    "GFP + XGBoost": 63.23,
    "Multi-GIN+EU": 64.79,
    "Multi-PNA": 64.59,
    "Multi-PNA+EU": 68.16,
}

fig, ax = plt.subplots(figsize=(9, 5.5))
names = list(published.keys())
vals = [published[k] for k in names]
bars = ax.barh(names, vals, color="#B0C4DE", label="published (papers)")
bars[-1].set_color("#4878CF")

# Overlay our runs from the RESULTS dict built through the notebook.
for i, (name, val) in enumerate(RESULTS.items()):
    ax.scatter([val], [len(names) - 1 if "PNA" in name else 3.5], zorder=3,
               color="#D65F5F", s=80, label=name if i == 0 else None)
    ax.annotate(f"{name}: {val:.1f}", (val, 0.4), color="#D65F5F")

ax.set_xlabel("minority-class F1 (%) on AMLworld HI-Small")
ax.set_title("Laundering detection: published results and this notebook's runs")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

print(json.dumps(RESULTS, indent=2))

## 7. Conclusions and what comes next

**What this notebook establishes once both runs are complete:**

1. A leakage-safe tabular baseline with the evaluation protocol this domain requires
   (temporal split, minority F1, PR-AUC, recall at alert budget).
2. A single-seed replication attempt of the best published GNN configuration, with the
   published mean +- std as the acceptance band.
3. The measured gap between "transaction + history features" and "topology-aware
   models", which is the quantitative argument for graph ML in AML.

**Follow-up experiments, in order of value:**

- **The ablation ladder** (Section 5.3): one run per adaptation.
- **LI-Small generalization**: rerun on the harder variant; then reproduce the paper's
  transfer result (HI-trained PNA fine-tuned 5 epochs on LI recovers 27.4 F1 vs 0.0
  zero-shot).
- **Novel-typology holdout**: retrain with one laundering pattern removed, test on it.
- **Integration**: this model becomes the `get_model_score` tool inside the FinTrust
  Sentinel investigation agent; borderline-confidence alerts route to an LLM
  investigator for evidence gathering (the hybrid the Vanguard paper recommends).

**References**

- Altman et al., 2023. *Realistic Synthetic Financial Transactions for Anti-Money
  Laundering Models.* NeurIPS Datasets and Benchmarks. arXiv:2306.16424
- Egressy et al., 2024. *Provably Powerful Graph Neural Networks for Directed
  Multigraphs.* AAAI. arXiv:2306.11586
- IBM/Multi-GNN: https://github.com/IBM/Multi-GNN (Apache-2.0)
- Pirmorad, 2025. *Exploring the In-Context Learning Capabilities of LLMs for Money
  Laundering Detection in Financial Graphs.* arXiv:2507.14785

*All data in this project is synthetic (IBM AMLworld). Nothing here is a production
compliance system.*